# 114 setup: run, split, data, model (shared by every section)
`RUN` picks a src9 run; the split is the one src9 trains on (`src9/dataset.load_splits`). Data = a fixed random subset (`N_TRAIN` train samples +
up to `N_PER_EVAL` per eval split), spectra computed exactly like `dataset.process_x` (tukey, |fft| / std) and cached in `runs/_nb114_cache/<RUN>/`
(fp16 memmap, ~1.4 GB on disk, read lazily: CPU stays small). `df` has one row per sample, with a `split` column (`train` or `eval/<box>/<name>`).

The checkpoint was saved by an earlier version of `arch/transformer.py` (flat names `encoder.embed`, `encoder.lasers`, ...); the working tree groups
them into `encoder.freq_enc` / `encoder.laser_enc`. Same weights, so the keys are renamed on load. The per-box normalization stats (`cum_mean`,
`cum_std`) are buffers in the checkpoint.

In [1]:
import os, sys, json, math
os.environ["TQDM_DISABLE"] = "1"
from pathlib import Path
import numpy as np, pandas as pd, torch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src9").exists())
sys.path.insert(0, str(ROOT / "src9")); import dataset as D9
from arch.transformer import Transformer

RUN = "s9-transformer-vase-1k"
CKPT = "latest-rank0.pt"
METRIC = "iou@0.08"
REDUCER = "pca"            # pca | tsne | umap
N_TRAIN, N_PER_EVAL, SEED = 1000, 40, 0
BS = 32                    # one batch size and fp32 everywhere: bf16 results shift with the batch size (up to .09 per pixel)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
args = json.loads((ROOT / f"runs/{RUN}/args.json").read_text())
assert args["model"] == "transformer" and args["window"] == "tukey", args
CACHE = ROOT / f"runs/_nb114_cache/{RUN}"; CACHE.mkdir(parents=True, exist_ok=True)
print(RUN, args["max_duration"], DEV, torch.cuda.mem_get_info() if DEV == "cuda" else "")

s9-transformer-vase-1k 1000ep cuda (7590051840, 16608133120)


In [2]:
# the split src9 trained on -> a fixed subset -> df (one row per sample)
samples, idx = D9.load_splits()
metas, dirs = list(samples.values()), list(samples)
rng = np.random.default_rng(SEED)
pick = {int(i): "train" for i in rng.choice(idx["train"], N_TRAIN, replace=False)}
for name, ids in idx.items():
    if name == "train": continue
    for i in rng.choice(ids, min(N_PER_EVAL, len(ids)), replace=False): pick.setdefault(int(i), name)
order = sorted(pick)
boxes = sorted({m["box"] for m in metas})
objects = sorted({"+".join(sorted(metas[i]["objects"])) for i in idx["train"]}) + ["unseen"]   # as build_dataset: sets the object head size
df = pd.DataFrame([dict(i=i, split=pick[i], box=metas[i]["box"],
                        object="+".join(sorted(metas[i]["objects"])) or "empty", n_objects=metas[i]["n_objects"], speaker=metas[i]["speaker"],
                        experiment=metas[i]["experiment"], position_id=metas[i]["position_id"], sample_id=metas[i]["sample_id"], layout=metas[i]["layout"])
                   for i in order])
df["is_train"] = df.split == "train"
n_objects_all, max_k = sorted({m["n_objects"] for m in metas}), max(m["n_objects"] for m in metas)
sub = {dirs[i]: metas[i] for i in order}                  # only the picked samples' metadata; the rest (~0.8 GB of dicts) is dropped
lid = {(m["experiment"], m["without_lid_position_id"], m["speaker"]) for m in sub.values() if "without_lid_position_id" in m}
twins = {d: m for d, m in samples.items() if (m["experiment"], m["position_id"], m["speaker"]) in lid}   # process_y takes a with-lid sample's targets from these
del samples, metas, dirs; import gc; gc.collect()
print(df.split.value_counts().head(), len(df), "samples,", df.split.nunique(), "splits")

2026_09_27_gastronorm_four_objs: 6155 samples


2026_10_05_green_plastic_four_objs: 5736 samples


2026_10_11_cardboard_four_objs: 7228 samples


split                                          samples    positions
-------------------------------------------  ---------  -----------
train                                            11219         3032
eval/gastro/vase                                   135           45
eval/gastro/candle                                 156           52
eval/gastro/cube                                   147           49
eval/gastro/cube-unseen-speaker-baseline           105           35
eval/gastro/cube-unseen-speaker-unseen              35           35
eval/gastro/soap-dispenser                         146           49
eval/gastro/cube-vase                               27            9
eval/gastro/two-cubes                               42           14
eval/gastro/cube-with-lid                           15            5
eval/gastro/cube-without-lid                        15            5
eval/gastro/cylinder-1000g                          27            9
eval/gastro/cylinder-500g                       

split
train                                       1000
eval/gastro/cube-unseen-speaker-baseline      40
eval/gastro/cube                              40
eval/gastro/vase                              40
eval/gastro/unseen-position                   40
Name: count, dtype: int64 3178 samples, 66 splits


In [3]:
# spectra X (N,L,F,C) fp16 memmap, targets Y (N,64,64), coms (N,K,2); built once per (RUN, N_TRAIN, N_PER_EVAL, SEED)
tag = f"{N_TRAIN}_{N_PER_EVAL}_{SEED}"
fx, fy, fc = CACHE / f"X_{tag}.npy", CACHE / f"Y_{tag}.npy", CACHE / f"coms_{tag}.npy"
if not fc.exists():
    L, F_, C = D9.load_fft(*next(iter(sub.items())), "hann").shape
    X = np.lib.format.open_memmap(fx, "w+", np.float16, (len(order), L, F_, C))
    for j, x in enumerate(D9.process_x(sub, "tukey")): X[j] = x
    X.flush(); del X
    ys = D9.process_y(sub | twins, objects, boxes, max_k, 64, 64)[:len(order)]
    np.save(fy, np.stack([y["y"] for y in ys])); np.save(fc, np.stack([y["coms"] for y in ys]))
X = np.load(fx, mmap_mode="r")                     # (N,100,1170,2) fp16, not normalized (the model normalizes)
Y = np.load(fy); COMS = np.load(fc)
BOX = torch.tensor([boxes.index(b) for b in df.box])
df["area"] = (Y > 0.5).sum((1, 2))                                     # true mask area, px of 64x64
import warnings; warnings.filterwarnings("ignore", "Mean of empty slice")
c = np.where(df.n_objects.values[:, None, None] > np.arange(COMS.shape[1])[None, :, None], COMS, np.nan)
df["com_row"], df["com_col"] = np.nanmean(c[..., 0], 1), np.nanmean(c[..., 1], 1)   # mean CoM of the objects, box fractions (row 1 = laser wall)
print(X.shape, Y.shape)

(3178, 100, 1170, 2) (3178, 64, 64)


In [4]:
# model: the working-tree arch, the checkpoint's weights (old flat names -> freq_enc / laser_enc)
sd = torch.load(ROOT / f"runs/{RUN}/checkpoints/{CKPT}", map_location="cpu", weights_only=False)["state"]["model"]
REN = {"embed": "freq_enc.embed", "band_emb": "freq_enc.freq_band_emb", "box_emb": "freq_enc.box_emb", "freqs": "freq_enc.freqs", "pool_q": "freq_enc.pool_q",
       "pool_norm": "freq_enc.pool_norm", "pool_k": "freq_enc.pool_k", "pool_o": "freq_enc.pool_o", "lasers": "laser_enc.lasers", "to_emb": "laser_enc.to_emb"}
def ren(k):
    p = k.split(".")
    return ".".join([p[0], REN[p[1]], *p[2:]]) if p[0] == "encoder" and p[1] in REN else k
sd = {ren(k): v for k, v in sd.items()}
m0 = next(iter(sub.values()))
info = dict(n_laser_rows=m0["n_rows"], n_laser_cols=m0["n_cols"], n_freqs=X.shape[2], n_channels=X.shape[3], freq_range=(float(m0["min_freq"]), float(m0["max_freq"])),
            objects=objects, n_objects=n_objects_all, boxes=boxes,
            cum_mean=sd["encoder.cum_mean"], cum_var=sd["encoder.cum_std"].float() ** -2)
model = Transformer(info)
print(model.load_state_dict(sd, strict=True))
model = model.to(DEV).eval(); del sd
R, Cc = info["n_laser_rows"], info["n_laser_cols"]; L = R * Cc                      # 10 x 10 laser grid, laser l = (l // Cc, l % Cc)
NB = len(model.encoder.freq_band_logf); BPB = model.encoder.freq_enc.bins_per_band  # 19 freq bands of 64 bins
FREQS = np.linspace(*info["freq_range"], X.shape[2])
BAND_HZ = np.array([FREQS[min(b * BPB + BPB // 2, len(FREQS) - 1)] for b in range(NB)])   # band centres, Hz
xb = lambda ids: torch.from_numpy(X[ids].astype(np.float32)).to(DEV)               # raw (B,L,F,C) on the GPU
print(f"L={L} ({R}x{Cc}), {NB} bands of {BPB} bins ~{FREQS[BPB] - FREQS[0]:.0f} Hz")

<All keys matched successfully>


L=100 (10x10), 19 bands of 64 bins ~49 Hz


In [5]:
# per-sample metrics, (B,H,W) probabilities vs (B,H,W) truth -> (B,), nan for an empty box (as src9/utils.py iou08 / miss08)
THR = 0.08
from utils import moments
nan = lambda v, y: torch.where((y > 0.5).flatten(1).any(1), v, torch.nan)          # empty boxes don't score
def iou_at(t): return lambda p, y: nan(((p > t) & (y > 0.5)).flatten(1).sum(1) / ((p > t) | (y > 0.5)).flatten(1).sum(1).clamp_min(1), y)
def miss_at(t): return lambda p, y: nan((~((p > t) & (y > 0.5)).flatten(1).any(1)).float(), y)
METRICS = {"iou@0.08": iou_at(THR), "miss@0.08": miss_at(THR), "iou@0.5": iou_at(0.5),
           "soft_iou": lambda p, y: nan(torch.minimum(p, y).flatten(1).sum(1) / torch.maximum(p, y).flatten(1).sum(1).clamp_min(1e-6), y),
           "com_err": lambda p, y: nan((moments(p)[1] - moments(y)[1]).norm(dim=-1), y)}   # centroid distance, box fractions
HIGHER_BETTER = {"iou@0.08": True, "miss@0.08": False, "iou@0.5": True, "soft_iou": True, "com_err": False}
metric = METRICS[METRIC]

In [6]:
# baseline forward over everything (fp32; training evaluated under bf16); check against the run's own saved eval scores
@torch.no_grad()
def predict(ids, bs=BS):
    out = []
    for s in range(0, len(ids), bs):
        j = ids[s:s + bs]
        out.append(model({"fft": xb(j), "box": BOX[j].to(DEV)})["mask_pred"])
    return torch.cat(out)
P = predict(np.arange(len(df)))                                                  # (N,64,64) on the GPU
YT = torch.from_numpy(Y).to(DEV)
for k, f in METRICS.items(): df[k] = f(P, YT).cpu().numpy()
print(df.groupby(df.is_train)[list(METRICS)].mean().round(3))
import score
ev = df[~df.is_train]; one = ev.groupby("split")[METRIC].mean()
ref = score.per_sample(RUN, score.last_epoch(RUN), THR).assign(split=lambda d: "eval/" + d.box + "/" + d.split).groupby("split").iou.mean()
j = one.dropna().index.intersection(ref.index)
print(f"{METRIC} here vs run's saved eval (same splits, subset): {one[j].mean():.3f} vs {ref[j].mean():.3f}, per-split corr {np.corrcoef(one[j], ref[j])[0, 1]:.2f}")

          iou@0.08  miss@0.08  iou@0.5  soft_iou  com_err
is_train                                                 
False        0.464      0.139    0.458     0.458    0.072
True         0.910      0.004    0.977     0.911    0.004


iou@0.08 here vs run's saved eval (same splits, subset): 0.441 vs 0.441, per-split corr 1.00
